# Single-frequency lasers: DBR, DFB, external cavity, VCSEL {#sec-single-mode}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/single-mode.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/single-mode.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter4/laserlib.py', 'laserlib.py')
else:
    sys.path.insert(0, '..')                 # laserlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import laserlib as lz
lz.use_style()

*Slides 140–144 (distributed Bragg reflector and distributed feedback lasers, commercial devices, example on the DFB
wavelength), 145–146 (external cavity laser diodes) and 147–151 (vertical cavity surface emitting lasers).*

## Key relations

A Fabry–Perot laser diode oscillates on several modes, and hops between them (@sec-ld-structures). WDM systems (Ch. 2) need one
wavelength, stable to a fraction of 0.8 nm. The solution is a mirror whose reflectance depends on wavelength: a **Bragg
grating** (Ch. 1, Bragg mirrors; Ch. 2, fiber Bragg gratings). A corrugation of period $\Lambda$ reflects strongly only near the
Bragg wavelength (slide 140b)

$$
\lambda_B=\frac{2n\Lambda}{q},\qquad q=1,2,\dots\ \text{(order)}.
$$

* In a **DBR laser** (slide 140) the gratings replace the facets, outside the pumped region.
* In a **DFB laser** (slides 141–142) the grating runs along the whole active region: feedback is distributed. A uniform grating
  does not lase *at* $\lambda_B$ but on two modes just outside the stop band (slide 142),
  $$\lambda_m=\lambda_B\pm\frac{\lambda_B^2}{2nL}(m+1).$$
  An asymmetry (a facet coating, or a quarter-wave phase shift in the middle of the grating) selects one of them.
* An **external cavity laser** (slides 145–146) uses an AR-coated diode chip, a lens and an external grating or filter: long cavity,
  narrow linewidth (kHz), tunable.
* A **VCSEL** (slides 147–151) has a cavity about one wavelength long between two epitaxial DBR mirrors. The mode spacing is so large
  that only one longitudinal mode fits under the gain curve, but the single-pass gain is tiny, so the mirrors need $R>99$ %.

## Worked example: DFB wavelength (slide 144), recomputed

> DFB laser with corrugation period $\Lambda=0.22$ µm, grating length 400 µm, effective index 3.5, first order. Find the Bragg
> wavelength, the mode wavelengths and their separation.

In [ ]:
#| label: tbl-dfb
#| tbl-cap: "Slide 144 recomputed with Kasap's formula λ_m = λ_B ± λ_B²(m + 1)/(2nL), m = 0."
lB, lm, lp = lz.dfb_modes(0.22e-6, 3.5, 400e-6, m=[0])
pd.DataFrame([("Bragg wavelength λ_B = 2nΛ", f"{lB*1e6:.4f} µm"), ("mode below λ_B (m = 0)", f"{lm[0]*1e6:.5f} µm"),
              ("mode above λ_B (m = 0)", f"{lp[0]*1e6:.5f} µm"), ("separation of the two modes", f"{(lp - lm)[0]*1e9:.2f} nm")],
             columns=["quantity", "value"])

## Coupled-mode theory of a grating (going further on slides 140–142)

The slide formula treats the grating as a pair of mirrors. Coupled-mode theory (forward and backward waves coupled by the
grating with coefficient $\kappa=\pi\Delta n/\lambda$) gives the exact response of a uniform grating of length $L$, with or without
gain. Passive, it is the fiber Bragg grating of Ch. 2: a stop band of width $\Delta\lambda\approx\lambda_B^2\kappa/(\pi n)$ for
strong gratings, with peak reflectance $\tanh^2(\kappa L)$. With gain, the transmission diverges at the frequencies where the
structure can oscillate by itself: the DFB laser modes.

```{ojs}
//| echo: false
import {gratingRT, cabs2, dbrSpectrum} from "./laser_sim.js"
viewof gr = Inputs.form({
  kL: Inputs.range([0.2, 5], {value: 2, step: 0.05, label: "coupling strength κL"}),
  gL: Inputs.range([0, 4], {value: 0, step: 0.01, label: "net gain gL (power gain × length)"}),
  shift: Inputs.radio(["uniform grating", "λ/4 phase shift in the middle"], {value: "uniform grating", label: "grating"}),
  view: Inputs.radio(["reflectance |r|²", "transmission |t|² (log)"], {value: "reflectance |r|²", label: "plot"})
})
```

```{ojs}
//| echo: false
grOut = {
  const L = 400e-6, neff = 3.5, Lam = 0.22e-6, lB = 2*neff*Lam, dn = gr.kL*lB/(Math.PI*L), g = gr.gL/L, data = [];
  let best = {T: 0};
  for (let i = 0; i <= 1200; i++) {
    const wl = lB - 4e-9 + 8e-9*i/1200, {r, t} = gratingRT(wl, Lam, neff, dn, L, g, gr.shift.startsWith("λ") ? Math.PI/2 : 0);
    const d = {wl: wl*1e9, R: cabs2(r), T: cabs2(t)}; data.push(d); if (d.T > best.T) best = d;
  }
  return {data, lB, best, dlB: lB*lB/(2*neff*L)};
}
Plot.plot({
  width: Math.min(width, 1050), height: 290, x: {label: "wavelength (nm)"},
  y: gr.view.startsWith("refl") ? {label: "|r|²", grid: true} : {label: "|t|²", type: "log", grid: true},
  marks: [Plot.line(grOut.data, {x: "wl", y: gr.view.startsWith("refl") ? "R" : "T", stroke: "#4a3aa7", strokeWidth: 2}),
          Plot.ruleX([grOut.lB*1e9], {stroke: "#52514e", strokeDasharray: "3,3"}),
          Plot.ruleX([grOut.lB*1e9 - grOut.dlB*1e9, grOut.lB*1e9 + grOut.dlB*1e9], {stroke: "#eb6834", strokeDasharray: "2,4"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">λ<sub>B</sub> = ${(grOut.lB*1e9).toFixed(2)} nm (grey) · Kasap's modes λ<sub>B</sub> ± λ<sub>B</sub>²/2nL (orange) ·
peak reflectance ${d3.max(grOut.data, d => d.R).toFixed(4)} (passive tanh²κL = ${(Math.tanh(gr.kL)**2).toFixed(4)}) ·
largest transmission <b>${grOut.best.T.toExponential(2)}</b> at <b>${grOut.best.wl.toFixed(3)} nm</b></div>`
```

Things to try:

* With $gL=0$, increase $\kappa L$: the stop band widens and the reflectance approaches 1.
* Raise the gain with the uniform grating and the transmission plot: two peaks grow symmetrically **on each side** of the stop
  band. At the gain where they diverge, the laser oscillates on both: the two-mode problem of slide 142.
* Switch to the λ/4-shifted grating: a single peak grows **at** $\lambda_B$, in the middle of the stop band, and it diverges at a
  lower gain. All modern telecom DFB lasers use a λ/4 shift or an asymmetric facet coating.

In [ ]:
#| label: fig-dfb-threshold
#| fig-cap: "Peak transmission of an amplifying grating (κL = 2, L = 400 µm) versus gain. The first divergence marks the lasing threshold: gL ≈ 1.4 for the λ/4-shifted grating, on a single mode at λ_B, against gL ≈ 2.0 for the uniform grating, whose two edge modes reach threshold together. The later peaks are higher-order modes."
wl = np.linspace(1.536e-6, 1.544e-6, 6001)
dn = 2*1.54e-6/(np.pi*400e-6)
gs = np.linspace(0, 12000, 120)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for sh, lab in ((0, "uniform"), (np.pi/2, "λ/4 shifted")):
    Tm = [np.max(np.abs(lz.grating_rt(wl, 0.22e-6, 3.5, dn, 400e-6, g=g, shift=sh)[1])**2) for g in gs]
    ax.semilogy(gs*400e-6, Tm, label=lab)
ax.set(xlabel="gL", ylabel="max |t|²", ylim=(0.5, 1e6)); ax.legend(fontsize=8); plt.show()

## Linewidths in the commercial tables (slide 143)

Spectral widths are quoted in frequency or wavelength, $\delta\lambda=\lambda^2\delta\nu/c$:

In [ ]:
#| label: tbl-linewidth
#| tbl-cap: "Slide 143: conversion between δν and δλ. The table footnote should read 'fm is 10⁻¹⁵ m' (not s)."
items = [("DBR, 1063 nm", 1063e-9, 2e6, "8 fm"), ("DFB, 1063 nm", 1063e-9, 2e6, "8 fm"),
         ("DFB, 1550 nm", 1550e-9, 10e6, "0.08 pm"), ("EC laser, 1550 nm", 1550e-9, 50e3, "0.4 fm"),
         ("FP laser (slide 100), 1550 nm", 1550e-9, None, "1.3 nm")]
rows = []
for name, lam, dnu, ref in items:
    if dnu is None:
        rows.append({"laser": name, "δν": f"{lz.c0*1.3e-9/lam**2/1e9:.0f} GHz", "δλ": "1.3 nm", "slide": ref,
                     "coherence length c/πδν": f"{lz.c0/(np.pi*lz.c0*1.3e-9/lam**2)*1e3:.2f} mm"})
    else:
        rows.append({"laser": name, "δν": f"{dnu/1e6:g} MHz", "δλ": f"{lam**2*dnu/lz.c0*1e15:.2f} fm", "slide": ref,
                     "coherence length c/πδν": f"{lz.c0/(np.pi*dnu):.3g} m"})
pd.DataFrame(rows)

A single-frequency laser is more coherent than a Fabry–Perot laser by six orders of magnitude (Ch. 1, coherence length). The
remaining linewidth comes from spontaneous emission into the lasing mode: see
[Going further: laser linewidth](linewidth-noise.qmd).

## VCSEL mirrors (slides 147–151)

In a VCSEL the light crosses a gain region only a few quantum wells thick, about 30 nm. The single-pass gain is about
$e^{gd}\approx1.003$ for $g=1000$ cm⁻¹, so the mirrors must lose less than 0.3 % per pass. Epitaxial AlAs/GaAs quarter-wave stacks
(Ch. 1, Bragg mirrors) provide this:

```{ojs}
//| echo: false
viewof vc = Inputs.form({
  N: Inputs.range([5, 40], {value: 25, step: 1, label: "number of DBR pairs N"}),
  nH: Inputs.range([3.0, 3.7], {value: 3.52, step: 0.01, label: "high index (GaAs)"}),
  nL: Inputs.range([1.5, 3.4], {value: 2.95, step: 0.01, label: "low index (AlAs; ≈ 1.6 for oxidised AlOx)"}),
  dact: Inputs.range([5, 100], {value: 24, step: 1, label: "total QW thickness (nm)"}),
  Gr: Inputs.range([1.0, 2.0], {value: 1.8, step: 0.05, label: "longitudinal enhancement (standing wave)"})
})
```

```{ojs}
//| echo: false
vcOut = {
  const wl0 = 850e-9, spec = [];
  for (let i = 0; i <= 400; i++) { const wl = 700e-9 + 300e-9*i/400; spec.push({wl: wl*1e9, R: dbrSpectrum(wl, vc.N, vc.nH, vc.nL, 1.0, vc.nH, wl0)}); }
  const R = dbrSpectrum(wl0, vc.N, vc.nH, vc.nL, vc.nH, vc.nH, wl0);    // mirror seen from inside the cavity
  const d = vc.dact*1e-9, gth = Math.log(1/(R*R))/(2*d*vc.Gr)/100;      // cm^-1 (internal loss neglected)
  return {spec, R, gth};
}
Plot.plot({
  width: Math.min(width, 1050), height: 250, x: {label: "wavelength (nm)"}, y: {label: "DBR reflectance (from air)", domain: [0, 1], grid: true},
  marks: [Plot.line(vcOut.spec, {x: "wl", y: "R", stroke: "#2a78d6", strokeWidth: 2}), Plot.ruleX([850], {stroke: "#52514e", strokeDasharray: "3,3"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">reflectance at 850 nm, seen from the cavity: <b>${(100*vcOut.R).toFixed(4)} %</b> ·
threshold gain with two such mirrors: <b>${vcOut.gth.toFixed(0)} cm⁻¹</b> ${vcOut.gth < 3000 ? "(reachable with strained QWs)" : "(<b>too high</b>: add pairs)"}</div>`
```

In [ ]:
#| label: tbl-vcsel
#| tbl-cap: "Slide 149 with recomputed slope efficiency and wavelength–frequency conversion of the linewidth."
v = pd.DataFrame([(680, "<0.1", 0.046, 21, 0.4, 3, 2.8, 0.95, 0.32, "STM"), (680, "0.5", 0.04, 15, 2.5, 8, 2.3, 4, 0.7, "MTM"),
                  (850, "0.0002", 0.06, 17, 0.5, 2, 2.2, 0.75, 0.5, "STM"), (850, "<0.65", 0.06, 26, 1, 6, 1.9, 2, 0.4, "MTM")],
                 columns=["λ (nm)", "δλ (nm)", "dλ/dT (nm/K)", "Δθ (°)", "I_th (mA)", "I (mA)", "V (V)", "P_o (mW)", "slope, slide", "type"])
v["slope = P_o/(I − I_th)"] = (v["P_o (mW)"]/(v["I (mA)"] - v["I_th (mA)"])).round(2)
v["PCE %"] = (100*v["P_o (mW)"]/(v["I (mA)"]*v["V (V)"])).round(0)
v

The 850 nm single-mode VCSEL has $\delta\lambda=0.2$ pm, that is 83 MHz, and the thresholds are sub-mA: a VCSEL's active volume is
a hundred times smaller than an edge emitter's. Low cost (wafer-level testing, 2D arrays, slide 150) made VCSELs the source of
short-reach datacom links over multimode fiber (Ch. 2) and of face recognition and lidar modules.

::: {.callout-tip collapse="true"}
## Going further
- **Hole burning in DFB lasers.** In a λ/4-shifted DFB with large $\kappa L$ the light is concentrated near the phase shift and
  depletes the carriers there ("longitudinal spatial hole burning"), which can make the laser jump to a side mode at high power.
  This sets the optimum $\kappa L\approx1$–1.5.
- **Tunable lasers.** Sampled-grating DBR lasers combine two combs of reflection peaks with slightly different periods (vernier
  effect) and tune over 40 nm. Build the two comb spectra numerically and show how a small index change of one comb moves the
  coincidence by many nm.
- **External-cavity linewidth.** The Schawlow–Townes linewidth scales as $1/\tau_{ph}^2$. Lengthening the cavity from 300 µm (chip)
  to 3 cm (external) multiplies $\tau_{ph}$ by about 100 if the losses per pass are similar: estimate the linewidth reduction.
- **VCSEL oxide aperture.** Selective oxidation of an AlGaAs layer creates an AlOx aperture (n ≈ 1.6) that confines both current and
  light, the key to sub-mA thresholds.
:::

::: {.callout-note collapse="true"}
## Complements
- H. Kogelnik and C. V. Shank, "Coupled-wave theory of distributed feedback lasers", *J. Appl. Phys.* 43, 2327 (1972).
- K. Iga, "Surface-emitting laser — its birth and generation of new optoelectronics field", *IEEE J. Sel. Top. Quantum Electron.* 6,
  1201 (2000).
- Kasap, sec. 4.15–4.17; Coldren *et al.*, ch. 3 (mirrors, gratings) and 8 (tunable lasers).
:::

## Try it yourself

1. A DFB laser must emit at 1550.12 nm (ITU channel 34) with $n_\text{eff}=3.22$. What grating period is needed in first and in
   second order?
2. With the grating panel, find $\kappa L$ for which the uniform grating reflects 90 %. Compare with $\tanh^2\kappa L$.
3. How many AlAs/GaAs pairs give $R=99.9$ % at 850 nm on a GaAs substrate? And with AlOx/GaAs pairs?
4. Why does a VCSEL need a lower threshold *current* but a higher threshold *gain* than an edge emitter?